## Survey of AWOIAF section headings

Goal: every character page on AWOIAF is structured into sections (`History`, `Recent Events`, `Appearances`, `Descendants`, `Quotes`, `See also`, `References`, ...). The current `parse_affiliated` in `scrape_characters.ipynb` walks the entire article body and collects every internal link, regardless of section — so ancestors, descendants, and references get folded into the network alongside actual storyline interactions.

Before deciding which sections to keep when rebuilding the affiliated list, we need to know which headings exist and how often. This notebook:

1. Fetches every character page (slugs from `characters.csv`).
2. Extracts each `<h2>`/`<h3>`/`<h4>` heading (`mw-headline`) inside `mw-parser-output`.
3. Aggregates `heading → number of characters who have it`, split by heading level.

Result is shown inline; nothing is written to disk.

### Setup
Same browser User-Agent and session pattern as `scrape_characters.ipynb` — the wiki returns 403 on generic UAs, and as of late May 2026 also presents a Cloudflare JS challenge intermittently. If a fetch returns a `Just a moment...` page the cell will warn; re-run later from a session the wiki has seen recently.

In [2]:
from bs4 import BeautifulSoup
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from urllib.parse import quote
import pandas as pd
import requests
from tqdm.auto import tqdm

BASE = 'https://awoiaf.westeros.org'
PREFIX = '/index.php/'

headers = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
}

session = requests.Session()
session.headers.update(headers)


def slug_to_url(slug):
    return BASE + PREFIX + quote(slug, safe="_/(),'")

### Heading extraction
MediaWiki renders section headings as e.g. `<h2><span class="mw-headline" id="History">History</span></h2>`. We collect every such span inside the `mw-parser-output` div and return `(level, text)` pairs. Levels: `h2` = top-level section, `h3` = subsection, `h4` = sub-subsection.

In [3]:
CHALLENGE_MARKER = 'Just a moment...'


def extract_headings(slug):
    """Return list of (level:str, text:str) for one character page. Returns None on failure."""
    try:
        resp = session.get(slug_to_url(slug), timeout=20)
    except requests.RequestException:
        return None
    if resp.status_code != 200:
        return None
    if CHALLENGE_MARKER in resp.text[:1000]:
        return 'CHALLENGE'
    soup = BeautifulSoup(resp.content, 'html.parser')
    content = soup.find('div', class_='mw-parser-output')
    if content is None:
        return []
    out = []
    for h in content.find_all(['h2', 'h3', 'h4', 'h5']):
        span = h.find('span', class_='mw-headline')
        if span is None:
            continue
        text = span.get_text(strip=True)
        if text:
            out.append((h.name, text))
    return out

### Run the survey
Parallel fetch over all ~3,689 character slugs. Results are aggregated in-memory; nothing is written to disk.

Set `SAMPLE = None` for the full run, or to e.g. `500` for a quick first look. The frequency distribution stabilizes well before the full count.

In [4]:
characters = pd.read_csv('../csvs/characters.csv')
slugs = characters['ID'].tolist()

SAMPLE = None        # None = all characters; int = first N (use 300 for a quick read)
WORKERS = 8

if SAMPLE is not None:
    slugs = slugs[:SAMPLE]
print(f'Surveying {len(slugs)} character pages with {WORKERS} workers')

# heading_text -> Counter of levels it appeared at
level_counts = defaultdict(Counter)
# heading_text -> set of character slugs that have it (for drill-down later)
examples = defaultdict(list)
errors = []
challenges = []

with ThreadPoolExecutor(max_workers=WORKERS) as executor:
    for slug, result in tqdm(
        zip(slugs, executor.map(extract_headings, slugs)),
        total=len(slugs),
    ):
        if result is None:
            errors.append(slug)
            continue
        if result == 'CHALLENGE':
            challenges.append(slug)
            continue
        seen_here = set()  # one character contributes at most once per heading text
        for level, text in result:
            if text in seen_here:
                continue
            seen_here.add(text)
            level_counts[text][level] += 1
            if len(examples[text]) < 5:
                examples[text].append(slug)

print(f'\nDone. {len(errors)} request errors, {len(challenges)} Cloudflare challenges.')
if challenges:
    print(f'  Example challenged slugs: {challenges[:5]}')

Surveying 3690 character pages with 8 workers


  0%|          | 0/3690 [00:00<?, ?it/s]


Done. 0 request errors, 0 Cloudflare challenges.


### Overview table
Each heading text, the total number of characters with that heading, and the count per HTML level.

In [5]:
rows = []
for text, lvls in level_counts.items():
    rows.append({
        'heading': text,
        'characters': sum(lvls.values()),
        'h2': lvls.get('h2', 0),
        'h3': lvls.get('h3', 0),
        'h4': lvls.get('h4', 0),
        'h5': lvls.get('h5', 0),
    })

overview = pd.DataFrame(rows).sort_values('characters', ascending=False).reset_index(drop=True)
n_chars = len(slugs) - len(errors) - len(challenges)
overview['pct'] = (overview['characters'] / max(n_chars, 1) * 100).round(1)

print(f'{len(overview)} distinct heading texts found across {n_chars} parsed character pages.\n')
with pd.option_context('display.max_rows', 200, 'display.max_colwidth', 60):
    print(overview.head(60).to_string(index=False))

1304 distinct heading texts found across 3690 parsed character pages.

                   heading  characters   h2  h3  h4  h5  pct
                References        3673 3673   0   0   0 99.5
                   History        2066 2064   2   0   0 56.0
             Recent Events        1856 1856   0   0   0 50.3
                    Family        1427 1423   4   0   0 38.7
         A Feast for Crows         820    0 820   0   0 22.2
  Appearance and Character         802  802   0   0   0 21.7
      A Dance with Dragons         801    0 801   0   0 21.7
         A Storm of Swords         794    1 793   0   0 21.5
                    Quotes         621  620   1   0   0 16.8
          A Clash of Kings         595    1 594   0   0 16.1
                 Character         411  410   1   0   0 11.1
         A Game of Thrones         383    0 383   0   0 10.4
                Appearance         345  345   0   0   0  9.3
                     Notes         275  275   0   0   0  7.5
         Behin

### Top-level sections only (h2)
These are the major page divisions — what we'd actually use to scope link collection when rebuilding the affiliated list.

In [6]:
top_level = overview[overview['h2'] > 0].sort_values('h2', ascending=False).reset_index(drop=True)
top_level = top_level[['heading', 'h2', 'pct']].rename(columns={'h2': 'characters_with_h2'})
with pd.option_context('display.max_rows', 200):
    print(top_level.to_string(index=False))

                                              heading  characters_with_h2  pct
                                           References                3673 99.5
                                              History                2064 56.0
                                        Recent Events                1856 50.3
                                               Family                1423 38.7
                             Appearance and Character                 802 21.7
                                               Quotes                 620 16.8
                                            Character                 410 11.1
                                           Appearance                 345  9.3
                                                Notes                 275  7.5
                                    Behind the Scenes                 215  5.8
                                       External links                  82  2.2
                                       External Link

### Drill-down: example characters per heading
Pick a heading to see five characters that have it — useful for sanity-checking what content lives under each (e.g. is `Family` mostly the infobox cousin to `Descendants`, or its own thing?).

In [7]:
for h in ['History', 'Recent Events', 'Appearances', 'Descendants', 'Family', 'Quotes', 'See also', 'References', 'Notes']:
    ex = examples.get(h, [])
    n = sum(level_counts[h].values()) if h in level_counts else 0
    print(f'{h!r:20s}  ({n:4d} characters)  e.g. {ex[:5]}')

'History'             (2066 characters)  e.g. ['A_certain_man', 'Abelar_Hightower', 'Addam_of_Duskendale', 'Addam_Frey', 'Addam_Hightower']
'Recent Events'       (1856 characters)  e.g. ['A_certain_man', 'Addam_Marbrand', 'Adrack_Humble', 'Aegon_Frey_(son_of_Stevron)', 'Aegon_III_Targaryen']
'Appearances'         (   1 characters)  e.g. ['Marwyn_Belmore']
'Descendants'         (  25 characters)  e.g. ['Aegon_III_Targaryen', 'Aegon_V_Targaryen', 'Aemon_Targaryen_(son_of_Jaehaerys_I)', 'Aerion_Targaryen_(son_of_Daemion)', 'Alyn_Velaryon']
'Family'              (1427 characters)  e.g. ['Addam_Frey', 'Addam_Hightower', 'Addam_Marbrand', 'Addam_Osgrey', 'Addam_Velaryon']
'Quotes'              ( 621 characters)  e.g. ['A_certain_man', 'Addam_Osgrey', 'Addam_Velaryon', 'Aegon_IV_Targaryen', 'Aegor_Rivers']
'See also'            (  32 characters)  e.g. ['Alyssa_Arryn', 'Artys_I_Arryn', "Barra's_mother", 'Ben_Plumm', 'Brandon_the_Builder']
'References'          (3673 characters)  e.g. ['A_certa

### Sample the `Recent Events` section

Pick 10 random characters that have a `Recent Events` h2 and print the actual text underneath, bucketed by h3 sub-section. The hypothesis: the h3 titles correspond to book names (`A Game of Thrones`, `A Clash of Kings`, ...), which would let us know in which book(s) a character actually appears — i.e. the storyline signal we want for the network.

In [8]:
import random


def extract_recent_events(slug):
    """Return list of (level, title, text) for the Recent Events section and each h3 child.
    The first item is ('h2', 'Recent Events', lead_text_before_first_h3). Returns None if
    the page has no Recent Events section, fails to fetch, or hits the Cloudflare challenge."""
    try:
        resp = session.get(slug_to_url(slug), timeout=20)
    except requests.RequestException:
        return None
    if resp.status_code != 200 or CHALLENGE_MARKER in resp.text[:1000]:
        return None
    soup = BeautifulSoup(resp.content, 'html.parser')
    content = soup.find('div', class_='mw-parser-output')
    if content is None:
        return None
    target = None
    for h in content.find_all('h2'):
        span = h.find('span', class_='mw-headline')
        if span and span.get_text(strip=True).lower() == 'recent events':
            target = h
            break
    if target is None:
        return None
    # Walk siblings until the next h2; bucket paragraphs by the most recent h3.
    buckets = [('h2', 'Recent Events', [])]
    for sib in target.find_next_siblings():
        if sib.name == 'h2':
            break
        if sib.name == 'h3':
            span = sib.find('span', class_='mw-headline')
            title = span.get_text(strip=True) if span else sib.get_text(strip=True)
            buckets.append(('h3', title, []))
        elif sib.name in ('p', 'ul', 'ol'):
            text = sib.get_text(' ', strip=True)
            if text:
                buckets[-1][2].append(text)
    out = [(lvl, t, ' '.join(paras)) for lvl, t, paras in buckets if paras]
    return out or None


random.seed(42)
candidates = random.sample(slugs, min(len(slugs), 50))
samples = []
for slug in candidates:
    data = extract_recent_events(slug)
    if data:
        samples.append((slug, data))
    if len(samples) >= 10:
        break

print(f'Got Recent Events from {len(samples)} of {len(candidates)} sampled characters.\n')

SNIPPET = 400  # chars per subsection
for slug, data in samples:
    print('=' * 80)
    print(f'CHARACTER: {slug}')
    sub_titles = [t for lvl, t, _ in data if lvl == 'h3']
    print(f'Subsections ({len(sub_titles)}): {sub_titles}')
    print('=' * 80)
    for level, title, text in data:
        marker = '##' if level == 'h2' else '  ###'
        print(f'\n{marker} {title}')
        snippet = text if len(text) <= SNIPPET else text[:SNIPPET] + '...'
        print(f'    {snippet}')
    print()

Got Recent Events from 10 of 50 sampled characters.

CHARACTER: Books
Subsections (1): ['A Dance with Dragons']

  ### A Dance with Dragons
    Books is part of the Siege of Astapor . He plays dice with Old Bill Bone , Beans and Archibald Yronwood after the siege. [1]

CHARACTER: Satin
Subsections (4): ['A Clash of Kings', 'A Storm of Swords', 'A Feast for Crows', 'A Dance with Dragons']

  ### A Clash of Kings
    The wandering crow Conwy brings Satin and other recruits from a lord 's dungeon near Gulltown to Castle Black . Before the great ranging , Jon Snow watches the new recruits being trained by Ser Endrew Tarth . [5]

  ### A Storm of Swords
    A scared Satin helps defend the King's Tower at Castle Black from the wildlings , fighting alongside Jon during the attack on Castle Black . He pisses himself when the horns are first blown. Satin proves to be a valuable asset to the defense of the castle, however, firing crossbow quarrels and pouring boiling oil on wildlings . Satin hel

### Diagnostic: were any of the 50 candidates blocked?

The previous cell's loop breaks at 10 successes, so most of the 50 candidates were never tried. And `extract_recent_events` returns `None` for four different reasons (Cloudflare challenge, HTTP error, network error, no `Recent Events` section) without distinguishing them.

This cell re-tries all 50 candidates with the same seed and reports the breakdown.

In [9]:
from collections import Counter


def classify_recent_events(slug):
    """Return ('status', detail). Status is one of: ok, no_section, challenge, http_error, request_error, no_content_div."""
    try:
        resp = session.get(slug_to_url(slug), timeout=20)
    except requests.RequestException as e:
        return ('request_error', str(e)[:80])
    if CHALLENGE_MARKER in resp.text[:1000]:
        return ('challenge', resp.status_code)
    if resp.status_code != 200:
        return ('http_error', resp.status_code)
    soup = BeautifulSoup(resp.content, 'html.parser')
    content = soup.find('div', class_='mw-parser-output')
    if content is None:
        return ('no_content_div', None)
    for h in content.find_all('h2'):
        span = h.find('span', class_='mw-headline')
        if span and span.get_text(strip=True).lower() == 'recent events':
            return ('ok', None)
    return ('no_section', None)


random.seed(42)
candidates = random.sample(slugs, min(len(slugs), 50))

results_by_status = Counter()
per_slug = []
with ThreadPoolExecutor(max_workers=8) as ex:
    for slug, (status, detail) in zip(candidates, ex.map(classify_recent_events, candidates)):
        results_by_status[status] += 1
        per_slug.append((slug, status, detail))

print('Outcome breakdown across all 50 candidates:')
for status, n in results_by_status.most_common():
    print(f'  {status:18s} {n}')

# Show the non-ok ones so you can spot patterns (e.g. all challenges, all 404s on disambig pages)
print('\nNon-ok slugs:')
for slug, status, detail in per_slug:
    if status != 'ok':
        suffix = f'  ({detail})' if detail else ''
        print(f'  [{status}] {slug}{suffix}')

Outcome breakdown across all 50 candidates:
  ok                 27
  no_section         23

Non-ok slugs:
  [no_section] Pol
  [no_section] Alfred_Broome
  [no_section] Frynne
  [no_section] Ellard_Crane
  [no_section] Sargoso_Saan
  [no_section] Bellena_Hawick
  [no_section] Narha_Otherys
  [no_section] John_II_Gardener
  [no_section] Alyn_Ashford
  [no_section] Alton_Greyjoy
  [no_section] Benifer
  [no_section] Drazenko_Rogare
  [no_section] Lyonel_Baratheon
  [no_section] Michael_Crowell
  [no_section] Denys_Harte
  [no_section] Qarlton_III_Durrandon
  [no_section] Joffrey_Doggett
  [no_section] Gallard
  [no_section] Aegon_Targaryen_(son_of_Aenys_I)
  [no_section] Tom_Turnip
  [no_section] Robin_Shaw
  [no_section] Galladon_Tarth
  [no_section] Hagon_Hoare


### Sanity check: subsections of `Family`

Same shape as the Recent Events sampler, but targets the `Family` h2. The point is to confirm where `Ancestors` and `Descendants` actually live, and to see what other subsections (e.g. `Siblings`, `Children`, `Family tree`) appear so the network-rebuild filter can drop them cleanly. Ends with an aggregated count of every h3 subsection title seen across the 10 samples.

In [10]:
def extract_family(slug):
    """Return list of (level, title, text) for the Family section and each h3 child.
    Returns None if the page has no Family section, fails to fetch, or hits Cloudflare."""
    try:
        resp = session.get(slug_to_url(slug), timeout=20)
    except requests.RequestException:
        return None
    if resp.status_code != 200 or CHALLENGE_MARKER in resp.text[:1000]:
        return None
    soup = BeautifulSoup(resp.content, 'html.parser')
    content = soup.find('div', class_='mw-parser-output')
    if content is None:
        return None
    target = None
    for h in content.find_all('h2'):
        span = h.find('span', class_='mw-headline')
        if span and span.get_text(strip=True).lower() == 'family':
            target = h
            break
    if target is None:
        return None
    buckets = [('h2', 'Family', [])]
    for sib in target.find_next_siblings():
        if sib.name == 'h2':
            break
        if sib.name == 'h3':
            span = sib.find('span', class_='mw-headline')
            title = span.get_text(strip=True) if span else sib.get_text(strip=True)
            buckets.append(('h3', title, []))
        elif sib.name in ('p', 'ul', 'ol'):
            text = sib.get_text(' ', strip=True)
            if text:
                buckets[-1][2].append(text)
    out = [(lvl, t, ' '.join(paras)) for lvl, t, paras in buckets if paras]
    return out or None


random.seed(7)  # different seed than the Recent Events sample to vary the picks
candidates = random.sample(slugs, min(len(slugs), 50))
samples = []
for slug in candidates:
    data = extract_family(slug)
    if data:
        samples.append((slug, data))
    if len(samples) >= 10:
        break

print(f'Got Family from {len(samples)} characters.\n')

SNIPPET = 300
subsection_counts = Counter()
for slug, data in samples:
    print('=' * 80)
    print(f'CHARACTER: {slug}')
    sub_titles = [t for lvl, t, _ in data if lvl == 'h3']
    print(f'Subsections ({len(sub_titles)}): {sub_titles}')
    print('=' * 80)
    for level, title, text in data:
        marker = '##' if level == 'h2' else '  ###'
        print(f'\n{marker} {title}')
        snippet = text if len(text) <= SNIPPET else text[:SNIPPET] + '...'
        print(f'    {snippet}')
        if level == 'h3':
            subsection_counts[title] += 1
    print()

print('=' * 80)
print(f'All distinct h3 subsections of Family across {len(samples)} samples:')
print('=' * 80)
for title, n in subsection_counts.most_common():
    print(f'  {n:3d}  {title}')

Got Family from 3 characters.

CHARACTER: Jocelyn_Swyft
Subsections (0): []

## Family
    It is unclear how Jocelyn is related to the other members of House Swyft .

CHARACTER: Brienne_Tarth
Subsections (1): ['Descent']

  ### Descent
    Previous speculation on Brienne's descent included that Brienne descends from Ser Duncan the Tall , a former Lord Commander of the Kingsguard during the reign of King Aegon V Targaryen . An argument often raised was that a shield bearing Duncan's personal arms was in the armory at Evenfall Hall , an...

CHARACTER: Walder_Frey
Subsections (1): ['Descendants']

  ### Descendants
    Walder has outlived seven wives, and is currently married to his eighth wife. He has over a hundred descendants, both trueborn and baseborn. He has had twenty-two trueborn sons and seven trueborn daughters from his marriages, with an unknown number of bastard sons and daughters. Descendants of his f...

All distinct h3 subsections of Family across 3 samples:
    1  Descent


In [11]:
def family_subsections(slug):
    try:
        resp = session.get(slug_to_url(slug), timeout=20)
    except requests.RequestException:
        return []
    if resp.status_code != 200 or CHALLENGE_MARKER in resp.text[:1000]:
        return []
    soup = BeautifulSoup(resp.content, 'html.parser')
    content = soup.find('div', class_='mw-parser-output')
    if content is None:
        return []
    target = None
    for h in content.find_all('h2'):
        span = h.find('span', class_='mw-headline')
        if span and span.get_text(strip=True).lower() == 'family':
            target = h
            break
    if target is None:
        return []
    titles = []
    for sib in target.find_next_siblings():
        if sib.name == 'h2':
            break
        if sib.name == 'h3':
            span = sib.find('span', class_='mw-headline')
            t = span.get_text(strip=True) if span else sib.get_text(strip=True)
            if t:
                titles.append(t)
    return titles


counts = Counter()
with ThreadPoolExecutor(max_workers=8) as ex:
    for titles in tqdm(ex.map(family_subsections, slugs), total=len(slugs)):
        counts.update(titles)

for title, _ in counts.most_common():
    print(title)

  0%|          | 0/3690 [00:00<?, ?it/s]

Descendants
Ancestors
House Lannister
Siblings
House Baratheon
Children
House Stark
Marbrand tree
Lannister tree
Patrilineal descent
Ancestry
Arryn tree
Targaryen tree
Descent
House Tully
Daena's Blackfyre descendants
Paternal (Baratheon) tree
Maternal (Florent) tree
House Royce
